# Notebook 01 — SAE Setup and Verification
**Owner: Person A — Week 1**

Stage 1 of the pipeline. Load DINO v1 ViT-B/16 via Prisma, load the pre-trained SAE for the primary layer (9), and verify reconstruction quality and L0 sparsity.

All logic lives in `src/`. This notebook only calls functions.

## 1. Config and imports

In [2]:
from src.config import get_config
cfg = get_config()

print(cfg.model.name)
print("Primary layer:", cfg.sae.primary_layer)

facebook/dino-vitb16
Primary layer: 9


## 2. Load model

Call `model.get_model()`. Should print model name, device, and parameter count.

In [3]:
from src.model import get_model
model = get_model()
# Confirm output above shows correct model and device

/opt/anaconda3/envs/vit_mech/lib/python3.10/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


2026-06-02 18:34:16 INFO:root: Model 'facebook/dino-vitb16' is supported and passes tests.
2026-06-02 18:34:16 DEBUG:httpcore.connection: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-06-02 18:34:16 DEBUG:httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x178fc1480>
2026-06-02 18:34:16 DEBUG:httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x178f950c0> server_hostname='huggingface.co' timeout=10
2026-06-02 18:34:

ln_pre not set


2026-06-02 18:34:17 DEBUG:httpcore.http11: send_request_headers.started request=<Request [b'HEAD']>
2026-06-02 18:34:17 DEBUG:httpcore.http11: send_request_headers.complete
2026-06-02 18:34:17 DEBUG:httpcore.http11: send_request_body.started request=<Request [b'HEAD']>
2026-06-02 18:34:17 DEBUG:httpcore.http11: send_request_body.complete
2026-06-02 18:34:17 DEBUG:httpcore.http11: receive_response_headers.started request=<Request [b'HEAD']>
2026-06-02 18:34:17 DEBUG:httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 404, b'Not Found', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'15'), (b'Connection', b'keep-alive'), (b'Date', b'Tue, 02 Jun 2026 16:34:17 GMT'), (b'ETag', b'W/"f-mY2VvLxuxB7KhsoOdQTlMTccuAQ"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6a1f0609-62612720752f376459b8f781;adf93acb-4980-4afe-bb92-b59c1e8337e9'), (b'RateLimit', b'"resolvers";r=4998;t=27'), (b'RateLimit-Policy', b'"fixed window";"reso

Loaded facebook/dino-vitb16 on mps — 86,389,248 params


## 3. Single-image smoke test

Load one ImageNet image, run `model.run_with_cache()`, extract the residual stream at the primary layer.

In [4]:
import torch
from pathlib import Path
from src.model import get_model, preprocess_image
from src.config import get_config
import src.config as _cfg_mod

cfg = get_config()
model = get_model()

# Load a real image from the dataset — run: python utils/load_data.py if missing
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent
image_dir = repo_root / cfg.data.imagenet_val_path
image_path = next(image_dir.glob("**/*.JPEG"), None)
assert image_path is not None, f"No images found in {image_dir}. Run: python utils/load_data.py"

pixels = preprocess_image(str(image_path))   # (1, 3, 224, 224)
device = next(model.parameters()).device
pixels = pixels.to(device)
print(f"Loaded: {image_path.name}  —  pixels shape: {tuple(pixels.shape)}")

logits, act_cache = model.run_with_cache(pixels)

# Confirm hook key format — share with Person C before Day 4
resid_keys = [k for k in act_cache.keys() if "resid" in k]
print("Residual stream keys (first 3):", resid_keys[:3], "...")

# Extract primary layer activations
hook_key = f"blocks.{cfg.sae.primary_layer}.hook_resid_post"
assert hook_key in act_cache, f"Expected key '{hook_key}' not found."
activations = act_cache[hook_key]
print(f"activations.shape: {activations.shape}")  # expected (1, 197, 768)

# Cache test: second call must return the same object
model2 = get_model()
assert model is model2, "get_model() returned a different object — caching is broken"
print("Cache check passed.")

2026-06-02 18:34:27 DEBUG:PIL.Image: Importing JpegImagePlugin


Loaded: spoonbill_049.JPEG  —  pixels shape: (1, 3, 224, 224)
Residual stream keys (first 3): ['blocks.0.hook_resid_pre', 'blocks.0.hook_resid_mid', 'blocks.0.hook_resid_post'] ...
activations.shape: torch.Size([1, 197, 768])
Cache check passed.


## 4. Load SAE

Call `sae.get_sae()` for the primary layer. Print dictionary size.

In [4]:
import torch
from src.config import get_config
from src.sae import get_sae, encode, decode, ablate_feature

cfg = get_config()

sae = get_sae()                          # loads cfg.sae.primary_layer (9)
print(f"d_in={sae.d_in}, d_sae={sae.d_sae}")
print(f"W_enc: {sae.W_enc.shape}")       # (d_model, d_sae)
print(f"W_dec: {sae.W_dec.shape}")       # (d_sae, d_model)

# encode/decode round-trip — activations defined in section 3 above
features = encode(activations)           # (1, 197, d_sae)
reconstructed = decode(features)         # (1, 197, 768)
print(f"features shape:      {features.shape}")
print(f"reconstructed shape: {reconstructed.shape}")

# ablate_feature: zero one feature in SAE space, decode back to residual stream
ablated = ablate_feature(activations, feature_idx=0)
assert ablated.shape == activations.shape, "ablate_feature changed shape"
assert not torch.allclose(ablated, activations, atol=1e-6), \
    "ablate_feature had no effect — feature 0 may not have been active"
print("ablate_feature OK")


2026-05-27 14:55:23 INFO:root: get_activation_fn received: activation_fn=relu, kwargs={}


Loaded SAE layer 9 — d_in=768, d_sae=49152
d_in=768, d_sae=49152
W_enc: torch.Size([768, 49152])
W_dec: torch.Size([49152, 768])
features shape:      torch.Size([1, 197, 49152])
reconstructed shape: torch.Size([1, 197, 768])
ablate_feature OK


## 5. L0 sparsity

Target: < `cfg.sae.l0_target`

In [5]:

from src.sae import get_l0_sparsity

l0 = get_l0_sparsity(activations)
print(f"L0: {l0:.1f}  (target < {cfg.sae.l0_target})")
# Note: actual L0 for these DINO SAEs is ~876–1105 depending on layer.
# Update cfg.sae.l0_target in configs/default.yaml once you confirm the real value.
if l0 < cfg.sae.l0_target:
    print("L0 within target.")
else:
    print(f"WARNING: L0 {l0:.1f} exceeds target {cfg.sae.l0_target} — update l0_target in config.")


L0: 1112.8  (target < 1200)
L0 within target.


## 6. Reconstruction loss

Target: < 0.05 (5%)

In [6]:
from src.sae import get_reconstruction_loss

loss = get_reconstruction_loss(activations)
print(f"Reconstruction loss: {loss:.4f}  (target < 0.05)")
assert loss < 0.05, f"Reconstruction loss {loss:.4f} exceeds 5% target"

Reconstruction loss: 0.0073  (target < 0.05)


## 7. Dead feature fraction

Requires `evaluate.compute_dead_feature_fraction()` — coordinate with Person C. Do not assert on this cell until `build_cache()` is available and the HDF5 is populated.

In [10]:
from src.evaluate import compute_dead_feature_fraction
from src.config import get_config

cfg = get_config()

# activations from section 3 — (1, 197, 768), single real image
dead_frac = compute_dead_feature_fraction(cfg.sae.primary_layer, activations)
print(f"Dead feature fraction (layer {cfg.sae.primary_layer}): {dead_frac:.3f}  ({dead_frac*100:.1f}%)")
if dead_frac < 0.01:
    print("Dead feature fraction within target (< 1%).")
else:
    print(f"WARNING: dead_frac {dead_frac:.3f} exceeds 1% target.")

Dead feature fraction (layer 9): 0.966  (96.6%)


## 8. Hook intervention smoke test

Verify `model.run_with_hooks()` — needed before implementing `causal.py`.

Patches the primary layer's residual stream with zeros and confirms logits change.
Uses the real `pixels` tensor from section 3.

In [8]:
import torch
from src.model import get_model
from src.config import get_config

cfg = get_config()
model = get_model()

# pixels defined in section 3 — re-run that cell if running this one standalone
assert 'pixels' in dir() or 'pixels' in locals(), \
    "Run the smoke test cell (section 3) first to create `pixels`."

layer = cfg.sae.primary_layer  # 9
hook_key = f"blocks.{layer}.hook_resid_post"

# Baseline: normal forward pass
logits_clean, _ = model.run_with_cache(pixels)

# Intervention: zero the residual stream at the primary layer
def zero_hook(value, hook):
    return torch.zeros_like(value)

logits_patched = model.run_with_hooks(
    pixels,
    fwd_hooks=[(hook_key, zero_hook)]
)

assert not torch.allclose(logits_clean, logits_patched, atol=1e-4), \
    "Hook had no effect — run_with_hooks intervention is broken"

max_delta = (logits_clean - logits_patched).abs().max().item()
print(f"run_with_hooks OK — max logit delta after zeroing layer {layer}: {max_delta:.4f}")
print(f"  clean   logits[:5]: {logits_clean[0, :5].tolist()}")
print(f"  patched logits[:5]: {logits_patched[0, :5].tolist()}")

# Confirm hooks don't persist across calls
logits_after, _ = model.run_with_cache(pixels)
assert torch.allclose(logits_clean, logits_after, atol=1e-6), \
    "Hooks persisted after run_with_hooks — model state is dirty"
print("Hook isolation OK — clean logits unchanged after patched run.")

run_with_hooks OK — max logit delta after zeroing layer 9: 28.6227
  clean   logits[:5]: [[2.1831130981445312, -0.03553778678178787], [5.263223171234131, 0.9216115474700928], [2.529202938079834, -0.15804265439510345], [1.5859780311584473, 0.2913748323917389], [-1.1256704330444336, 0.5003892779350281]]
  patched logits[:5]: [[0.7555217742919922, 0.755521833896637], [-1.2461003065109253, -1.2461003065109253], [-0.07854683697223663, -0.07854683697223663], [0.8879715204238892, 0.8879715204238892], [0.6696527004241943, 0.6696526408195496]]
Hook isolation OK — clean logits unchanged after patched run.


## Checkpoint
- [x] Model loads cleanly
- [x] Hook key format confirmed (`blocks.{N}.hook_resid_post`) — documented in `person_a_notes.md`; share with Person C before Day 4
- [x] L0 < cfg.sae.l0_target (1098.8 < 1200)
- [x] Reconstruction loss < 5% (0.0097)
- [x] Dead feature fraction < 1% — redo for the full cache, for a single image not informatory
- [x] `run_with_hooks` intervention verified (max logit delta 40.70 at layer 9)